# Laboratorio 05 — RAG con búsqueda híbrida y re-ranker
### Python AI Developer 2026 · Capítulo 3: RAG

**Objetivo:** mejorar el pipeline RAG del Lab 04 con dos técnicas que aumentan la calidad del retrieval: búsqueda híbrida (densa + dispersa) y re-ranking.

**Prerequisito:** haber completado el Lab 04.

**Duración estimada:** 90 min  
**Setup adicional:** `uv add rank-bm25`

---
## Parte 1 — Las limitaciones del RAG básico

El retrieval por similitud semántica (embeddings) del Lab 04 tiene un problema:

**Caso 1 — Términos exactos:** si el usuario pregunta por "kubectl rollout undo", el retriever semántico puede traer documentos sobre kubernetes en general en lugar del chunk exacto que contiene ese comando.

**Caso 2 — Irrelevancia posicional:** los LLMs tienden a ignorar información en el medio del contexto (paper "Lost in the Middle", 2023). Si el chunk más relevante queda en el medio, el modelo puede ignorarlo.

Las soluciones:
- **Búsqueda híbrida:** combinar búsqueda semántica (densa) con búsqueda por palabras clave (BM25/dispersa)
- **Re-ranking:** reordenar los chunks recuperados poniendo los más relevantes primero

In [3]:
# reutilizar la colección de documentos del Lab 04
# si no tienes la carpeta ./docs_techcorp/, vuelve a ejecutar la Parte 2 del Lab 04
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import glob
from rank_bm25 import BM25Okapi
import numpy as np

todos_los_docs = []
for ruta in sorted(glob.glob("./docs_techcorp/*.txt")):
    nombre = ruta.split("/")[-1]
    with open(ruta, "r", encoding="utf-8") as f:
        contenido = f.read()
    todos_los_docs.append(Document(page_content=contenido, metadata={"fuente": nombre}))

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(todos_los_docs)

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_lab05",
)

print(f"Documentos: {len(todos_los_docs)} | Chunks: {len(chunks)} | Vectorstore: {vectorstore._collection.count()} docs")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Documentos: 5 | Chunks: 13 | Vectorstore: 52 docs


---
## Parte 2 — BM25: búsqueda por palabras clave

BM25 (Best Match 25) es el algoritmo clásico de búsqueda por palabras clave. Es el motor detrás de motores de búsqueda como Elasticsearch.

A diferencia del retrieval semántico, BM25 busca coincidencias exactas de palabras. Esto lo hace mejor para:
- Términos técnicos específicos (nombres de comandos, códigos, etc.)
- Números y fechas exactas
- Acrónimos y siglas

In [4]:
# construir el índice BM25 sobre los mismos chunks
# BM25 trabaja con tokens (palabras), no con vectores
corpus_tokenizado = [chunk.page_content.lower().split() for chunk in chunks]
bm25 = BM25Okapi(corpus_tokenizado)


def buscar_bm25(query: str, k: int = 3):
    """Busca los k chunks más relevantes usando BM25 (palabras clave)."""
    tokens_query = query.lower().split()
    scores = bm25.get_scores(tokens_query)
    # obtener índices de los k mejores scores
    indices_top = np.argsort(scores)[::-1][:k]
    return [(chunks[i], scores[i]) for i in indices_top]


# probar BM25 con una pregunta de término técnico exacto
query_test = "kubectl rollout undo"
resultados_bm25 = buscar_bm25(query_test, k=3)

print(f"BM25 para: '{query_test}'")
for i, (doc, score) in enumerate(resultados_bm25):
    print(f"  [{i+1}] score={score:.3f}: '{doc.page_content[:100]}...'")

BM25 para: 'kubectl rollout undo'
  [1] score=6.173: 'ROLLBACK
Si un deploy causa incidentes:
1. Detectar el problema (Grafana o reporte de usuario)
2. No...'
  [2] score=0.000: 'SERVIDORES DE PRODUCCIÓN
TechCorp opera 3 servidores de producción en el datacenter de Lima:
- Serve...'
  [3] score=0.000: 'MANUAL DE INFRAESTRUCTURA — TechCorp Perú S.A.C....'


In [5]:
# comparar BM25 vs semántico para el mismo query
retriever_semantico = vectorstore.as_retriever(search_kwargs={"k": 3})

print("Comparación para query: 'kubectl rollout undo'")
print()
print("SEMÁNTICO (embeddings):")
for doc in retriever_semantico.invoke(query_test):
    print(f"  '{doc.page_content[:100]}...'")

print()
print("BM25 (palabras clave):")
for doc, score in buscar_bm25(query_test):
    print(f"  score={score:.2f}: '{doc.page_content[:100]}...'")

print()
print("Observar cuál recupera mejor el chunk que contiene el comando específico.")

Comparación para query: 'kubectl rollout undo'

SEMÁNTICO (embeddings):
  'ROLLBACK
Si un deploy causa incidentes:
1. Detectar el problema (Grafana o reporte de usuario)
2. No...'
  'ROLLBACK
Si un deploy causa incidentes:
1. Detectar el problema (Grafana o reporte de usuario)
2. No...'
  'ROLLBACK
Si un deploy causa incidentes:
1. Detectar el problema (Grafana o reporte de usuario)
2. No...'

BM25 (palabras clave):
  score=6.17: 'ROLLBACK
Si un deploy causa incidentes:
1. Detectar el problema (Grafana o reporte de usuario)
2. No...'
  score=0.00: 'SERVIDORES DE PRODUCCIÓN
TechCorp opera 3 servidores de producción en el datacenter de Lima:
- Serve...'
  score=0.00: 'MANUAL DE INFRAESTRUCTURA — TechCorp Perú S.A.C....'

Observar cuál recupera mejor el chunk que contiene el comando específico.


---
## Parte 3 — Búsqueda híbrida: combinar lo mejor de ambos

La búsqueda híbrida fusiona los resultados de BM25 y semántica usando un algoritmo de fusión de rankings (RRF — Reciprocal Rank Fusion). La idea es simple: un documento que aparece en el top de ambas búsquedas merece mayor confianza que uno que solo aparece en una.

In [6]:
def reciprocal_rank_fusion(resultados_lista: list, k: int = 60) -> list:
    """
    Fusiona múltiples listas de resultados usando Reciprocal Rank Fusion.
    k=60 es el valor estándar de la literatura.

    Para cada documento, su score RRF es la suma de 1/(k + posición) en cada lista.
    Documentos que aparecen bien rankeados en múltiples listas obtienen el mayor score.
    """
    scores = {}
    for resultados in resultados_lista:
        for posicion, doc in enumerate(resultados):
            # usar el contenido como clave única del documento
            clave = doc.page_content
            if clave not in scores:
                scores[clave] = {"doc": doc, "score": 0.0}
            scores[clave]["score"] += 1.0 / (k + posicion + 1)

    # ordenar por score descendente
    ordenados = sorted(scores.values(), key=lambda x: x["score"], reverse=True)
    return [item["doc"] for item in ordenados]


def busqueda_hibrida(query: str, k_final: int = 4) -> list:
    """Combina búsqueda semántica y BM25 para obtener los mejores resultados."""
    # búsqueda semántica
    docs_semanticos = retriever_semantico.invoke(query)

    # búsqueda BM25
    docs_bm25 = [doc for doc, _ in buscar_bm25(query, k=4)]

    # fusionar con RRF y retornar los top k_final
    docs_fusionados = reciprocal_rank_fusion([docs_semanticos, docs_bm25])
    return docs_fusionados[:k_final]


# probar la búsqueda híbrida
print("Búsqueda híbrida para: 'kubectl rollout undo'")
for i, doc in enumerate(busqueda_hibrida("kubectl rollout undo")):
    print(f"  [{i+1}] '{doc.page_content[:120]}...'")

Búsqueda híbrida para: 'kubectl rollout undo'
  [1] 'ROLLBACK
Si un deploy causa incidentes:
1. Detectar el problema (Grafana o reporte de usuario)
2. Notificar al canal #in...'
  [2] 'SERVIDORES DE PRODUCCIÓN
TechCorp opera 3 servidores de producción en el datacenter de Lima:
- Server-PROD-01: aplicació...'
  [3] 'MANUAL DE INFRAESTRUCTURA — TechCorp Perú S.A.C....'
  [4] 'ACCESO A SISTEMAS
El acceso a servidores de producción requiere:
1. Aprobación del Tech Lead del área
2. Autenticación d...'


---
## Parte 4 — Re-ranking

Incluso con búsqueda híbrida, el orden de los chunks importa. El paper "Lost in the Middle" (2023) mostró que los LLMs prestan más atención al principio y al final del contexto.

El re-ranking reordena los chunks recuperados para poner los más relevantes en las posiciones de mayor atención. Implementamos un re-ranker simple usando el propio LLM como juez.

In [7]:
import anthropic
import json

cliente = anthropic.Anthropic()


def reranker_llm(query: str, docs: list, top_k: int = 3) -> list:
    """
    Re-rankea documentos usando el LLM como juez de relevancia.
    Para cada documento asigna un score de 1 a 5 según qué tan relevante
    es para responder la pregunta.

    Nota: en producción se usaría un cross-encoder dedicado (ej: ms-marco)
    que es más eficiente. Aquí usamos el LLM para que sea más transparente.
    """
    scored_docs = []

    for doc in docs:
        prompt = f"""Evalúa qué tan relevante es este fragmento para responder la pregunta.
Responde SOLO con un número del 1 al 5 (5 = muy relevante, 1 = irrelevante).

Pregunta: {query}
Fragmento: {doc.page_content[:300]}
Score:"""

        r = cliente.messages.create(
            model="claude-haiku-4-5-20251001",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=5,
            temperature=0,
        )
        try:
            score = int(r.content[0].text.strip()[0])
        except:
            score = 1
        scored_docs.append((doc, score))

    # ordenar por score descendente
    scored_docs.sort(key=lambda x: x[1], reverse=True)
    return [doc for doc, _ in scored_docs[:top_k]]


print("Función de re-ranking definida")

Función de re-ranking definida


In [8]:
# probar el re-ranking
query_rerank = "¿Qué hacer después de un deploy fallido?"

docs_hibridos = busqueda_hibrida(query_rerank, k_final=5)
docs_rerankeados = reranker_llm(query_rerank, docs_hibridos, top_k=3)

print(f"Query: '{query_rerank}'")
print()
print("Antes del re-ranking (orden híbrido):")
for i, doc in enumerate(docs_hibridos[:3]):
    print(f"  [{i+1}] '{doc.page_content[:100]}...'")

print()
print("Después del re-ranking (orden por relevancia):")
for i, doc in enumerate(docs_rerankeados):
    print(f"  [{i+1}] '{doc.page_content[:100]}...'")

Query: '¿Qué hacer después de un deploy fallido?'

Antes del re-ranking (orden híbrido):
  [1] 'VENTANAS DE DEPLOY A PRODUCCIÓN
Los deploys a producción solo se permiten:
- Martes y jueves entre 1...'
  [2] 'ROLLBACK
Si un deploy causa incidentes:
1. Detectar el problema (Grafana o reporte de usuario)
2. No...'
  [3] 'PROCESO DE DEPLOY — TechCorp Perú S.A.C.

AMBIENTES
TechCorp mantiene 4 ambientes:
- desarrollo (dev...'

Después del re-ranking (orden por relevancia):
  [1] 'ROLLBACK
Si un deploy causa incidentes:
1. Detectar el problema (Grafana o reporte de usuario)
2. No...'
  [2] 'POST-MORTEM
Todo incidente P1 o P2 requiere un post-mortem en Confluence dentro de las 48 horas sigu...'
  [3] 'VENTANAS DE DEPLOY A PRODUCCIÓN
Los deploys a producción solo se permiten:
- Martes y jueves entre 1...'


---
## Parte 5 — Pipeline RAG mejorado completo

In [9]:
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [10]:
llm = ChatAnthropic(
    model="claude-haiku-4-5-20251001",
    temperature=0.1,
    max_tokens=500,
)

PROMPT_RAG = ChatPromptTemplate.from_template("""
Eres un asistente técnico de TechCorp. Responde usando ÚNICAMENTE el contexto dado.
Si la respuesta no está en el contexto, di: "No encuentro esa información en el manual."

Contexto:
{contexto}

Pregunta: {pregunta}
Respuesta:
""")


def pipeline_rag_mejorado(pregunta: str) -> str:
    """Pipeline RAG con búsqueda híbrida + re-ranking."""
    # paso 1: búsqueda híbrida
    docs_recuperados = busqueda_hibrida(pregunta, k_final=5)

    # paso 2: re-ranking
    docs_finales = reranker_llm(pregunta, docs_recuperados, top_k=3)

    # paso 3: construir contexto y generar respuesta
    contexto = "\n\n".join(doc.page_content for doc in docs_finales)
    prompt_final = PROMPT_RAG.format_messages(contexto=contexto, pregunta=pregunta)
    respuesta = llm.invoke(prompt_final)
    return respuesta.content


print("Pipeline RAG mejorado definido")

Pipeline RAG mejorado definido


In [11]:
# comparar el pipeline básico (Lab 04) vs el mejorado (Lab 05)
# para preguntas que el retrieval básico maneja mal

# pipeline básico del Lab 04
retriever_basico = vectorstore.as_retriever(search_kwargs={"k": 3})
pipeline_basico = (
    {"contexto": retriever_basico | (lambda docs: "\n\n".join(d.page_content for d in docs)),
     "pregunta": RunnablePassthrough()}
    | PROMPT_RAG
    | llm
    | StrOutputParser()
)

preguntas_comparacion = [
    "¿Cuál es el comando exacto para hacer rollback en kubernetes?",
    "¿Cuántos días se retienen los backups diarios?",
]

for pregunta in preguntas_comparacion:
    print(f"Pregunta: {pregunta}")
    print(f"  RAG básico:   {pipeline_basico.invoke(pregunta)[:300]}")
    print(f"  RAG mejorado: {pipeline_rag_mejorado(pregunta)[:300]}")
    print()

Pregunta: ¿Cuál es el comando exacto para hacer rollback en kubernetes?
  RAG básico:   El comando exacto para hacer rollback en Kubernetes es:

```
kubectl rollout undo deployment/app-prod
```

Este comando se ejecuta como parte del procedimiento de rollback cuando un deploy causa incidentes.
  RAG mejorado: El comando exacto para hacer rollback en Kubernetes es:

```
kubectl rollout undo deployment/app-prod
```

Este comando se ejecuta después de que el Tech Lead de turno apruebe el rollback, una vez detectado un problema en el deploy.

Pregunta: ¿Cuántos días se retienen los backups diarios?
  RAG básico:   Según el manual, los backups diarios se retienen durante **30 días**.
  RAG mejorado: Los backups diarios se retienen durante **30 días**.



---
## Parte 6 — Evaluación con métricas manuales

Comparamos los dos pipelines sobre el mismo conjunto de preguntas para medir la mejora.

In [12]:
eval_set = [
    {"pregunta": "¿Cuántos servidores de producción tiene TechCorp?",      "clave": "3"},
    {"pregunta": "¿Cada cuánto tiempo rotar las contraseñas?",              "clave": "90"},
    {"pregunta": "¿Qué gestor de contraseñas usa la empresa?",              "clave": "1Password"},
    {"pregunta": "¿Cuál es el comando de rollback en kubernetes?",           "clave": "kubectl rollout undo"},
    {"pregunta": "¿Qué herramienta usa TechCorp para distributed tracing?",  "clave": "Jaeger"},
]

resultados = {"basico": 0, "mejorado": 0}

for item in eval_set:
    r_basico   = pipeline_basico.invoke(item["pregunta"])
    r_mejorado = pipeline_rag_mejorado(item["pregunta"])

    ok_basico   = item["clave"].lower() in r_basico.lower()
    ok_mejorado = item["clave"].lower() in r_mejorado.lower()

    if ok_basico:   resultados["basico"]   += 1
    if ok_mejorado: resultados["mejorado"] += 1

    print(f"P: {item['pregunta'][:50]}...")
    print(f"  Básico: {'✓' if ok_basico else '✗'}  |  Mejorado: {'✓' if ok_mejorado else '✗'}")

n = len(eval_set)
print()
print(f"Accuracy básico:   {resultados['basico']}/{n} = {resultados['basico']/n*100:.0f}%")
print(f"Accuracy mejorado: {resultados['mejorado']}/{n} = {resultados['mejorado']/n*100:.0f}%")

P: ¿Cuántos servidores de producción tiene TechCorp?...
  Básico: ✗  |  Mejorado: ✓
P: ¿Cada cuánto tiempo rotar las contraseñas?...
  Básico: ✓  |  Mejorado: ✓
P: ¿Qué gestor de contraseñas usa la empresa?...
  Básico: ✓  |  Mejorado: ✓
P: ¿Cuál es el comando de rollback en kubernetes?...
  Básico: ✓  |  Mejorado: ✓
P: ¿Qué herramienta usa TechCorp para distributed tra...
  Básico: ✓  |  Mejorado: ✓

Accuracy básico:   4/5 = 80%
Accuracy mejorado: 5/5 = 100%
